In [122]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [123]:
X = pd.read_csv("train.csv")
y = X.pop("label")

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [124]:
class Flatten:
    def __init__(self):
        pass

    def __call__(self, x):
        self.curr = x.reshape(x.shape[0], -1)
        
        return self

    def backward(self):
        pass

    def grad_descent(self, alpha):
        pass

In [125]:
class ReLU:
    def __init__(self):
        pass

    def __call__(self, prev_linear):
        self.curr = np.maximum(prev_linear.curr, 0)
        self.prev_linear = prev_linear
        
        return self
    
    def backward(self):
        self.prev_linear.dcurr = self.dcurr * (self.prev_linear.curr > 0)

        self.prev_linear.backward()

    def grad_descent(self, alpha):
        self.prev_linear.grad_descent(alpha)

In [ ]:
class Linear:
    def __init__(self, n_in, n_out):
        self.w = np.random.randn(n_in, n_out) * np.sqrt(2 / n_in)
        self.b = np.zeros(n_out)

    def __call__(self, prev_linear):
        self.prev_linear = prev_linear
        self.curr = self.prev_linear.curr @ self.w + self.b

        return self
    
    def backward(self):
        self.dw = self.prev_linear.curr.T @ self.dcurr / self.dcurr.shape[0]
        self.db = np.sum(self.dcurr, axis=0) / self.dcurr.shape[0]
        self.prev_linear.dcurr = self.dcurr @ self.w.T
        
        self.prev_linear.backward()

    def grad_descent(self, alpha):
        self.w -= self.dw*alpha
        self.b -= self.db*alpha

        self.prev_linear.grad_descent(alpha)


In [127]:
def softmax(z):
    z = np.asarray(z)
    e_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return e_z / np.sum(e_z, axis=1, keepdims=True)

class CrossEntropy():
    def __init__(self, logits_linear, y):
        self.y = y
        self.logits_linear = logits_linear
        self.y_cap = softmax(logits_linear.curr)

        self.value = -np.sum(self.y*np.log(self.y_cap + 1e-15))

    def backward(self):
        self.logits_linear.dcurr = self.y_cap - self.y
        
        self.logits_linear.backward()

    def grad_descent(self, alpha):
        self.logits_linear.grad_descent(alpha)

In [ ]:
def one_hot(y, num_classes=10):
    res = np.zeros((len(y), num_classes))
    res[np.arange(len(y)), y] = 1
    return res

class NN:
    def __init__(self):
        self.flatten = Flatten()
        self.fc1 = Linear(28*28, 64)
        self.relu12 = ReLU()
        self.fc2 = Linear(64, 32)
        self.relu23 = ReLU()
        self.fc3 = Linear(32, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu12(self.fc1(x))
        x = self.relu23(self.fc2(x))
        x = self.fc3(x)

        return x

    def predict(self, Xt):
        Xt = np.asarray(Xt, dtype=np.float64) / 255.0
        logits_linear = self.forward(Xt)
        return np.argmax(logits_linear.curr, axis=1)

    def fit(self, Xt, yt, epochs=50, alpha=0.1, batch_size=32, verbose=0):
        Xt = np.asarray(Xt, dtype=np.float64)/255.0
        yt = one_hot(yt)
        
        n = Xt.shape[0]

        rng = np.random.default_rng(42)

        for epoch in range(epochs):
            indices = rng.permutation(n)
            Xt = Xt[indices]
            yt = yt[indices]

            l = 0

            for start in range(0, n, batch_size):
                end = min(start + batch_size, n)
                X_batch = Xt[start:end]
                y_batch = yt[start:end]

                logits_linear = self.forward(X_batch)
                loss = CrossEntropy(logits_linear, y_batch)
                loss.backward()
                loss.grad_descent(alpha)

                l += loss.value

            if verbose and ((epoch+1)%verbose == 1 or epoch+1==epochs):
                print(f"epoch {epoch+1}/{epochs}. Loss: {l/n}")

        print("done.")

In [129]:
nn = NN()

nn.fit(X_train, y_train, alpha=0.1, epochs=50, batch_size=64, verbose=10)

epoch 1/50. Loss: 0.4631211253940407
epoch 11/50. Loss: 0.04776114220079194
epoch 21/50. Loss: 0.0133761529396091
epoch 31/50. Loss: 0.0027659512333199254
epoch 41/50. Loss: 0.001366526362695889
epoch 50/50. Loss: 0.000916317656510203
done.


In [134]:
def accuracy(y_cap, y):
    return np.mean(y_cap==y)

y_cap = nn.predict(X_val)
print(accuracy(y_cap, y_val))

0.9704761904761905


In [135]:
X_test = pd.read_csv("test.csv")
y_test_cap = nn.predict(X_test)

res = pd.DataFrame({
    "ImageId": range(1, len(y_test_cap)+1),
    "Label": y_test_cap
})

res.to_csv("sumbission.csv", index=False)